# Qwen-Image-Edit-2509 Inference on Trainium2

This notebook demonstrates how to run the Qwen-Image-Edit-2509 model on AWS Trainium2 (trn2) instances using Neuron SDK.

## Model Overview

Qwen-Image-Edit-2509 is an advanced image editing model that supports:
- Single and multi-image editing (1-3 input images)
- Enhanced identity preservation for persons and products
- Text editing capabilities
- Native ControlNet support

## Prerequisites

- AWS Trainium2 instance (trn2.48xlarge recommended)
- Neuron SDK installed
- Compiled models (run `compile.sh` first)

## Step 1: Install Dependencies

In [1]:
import os

WORK_DIR = "/home/ubuntu/hf_pretrained_qwen_image_edit"
os.chdir(WORK_DIR)
print(f"Working directory: {os.getcwd()}")

# Add to Python path so imports work
import sys
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)
    print(f"Added {WORK_DIR} to sys.path")

Working directory: /home/ubuntu/hf_pretrained_qwen_image_edit
Added /home/ubuntu/hf_pretrained_qwen_image_edit to sys.path


In [2]:
%%bash
# Update all MODEL_ID references from 2509 to 2511
# The 2511 model is architecturally identical to 2509 (weight-only update)

cd /home/ubuntu/hf_pretrained_qwen_image_edit

echo "Updating MODEL_ID references from 2509 to 2511..."

# Replace in all Python and shell files
find . -type f \( -name '*.py' -o -name '*.sh' \) -exec \
    sed -i 's/Qwen-Image-Edit-2509/Qwen-Image-Edit-2511/g' {} +

UPDATED=$(grep -r "Qwen-Image-Edit-2511" --include='*.py' --include='*.sh' -l | wc -l)
REMAINING=$(grep -r "Qwen-Image-Edit-2509" --include='*.py' --include='*.sh' -l | wc -l)
echo "Files with 2511 references: $UPDATED"
echo "Files with 2509 references remaining: $REMAINING"

Updating MODEL_ID references from 2509 to 2511...
Files with 2511 references: 25
Files with 2509 references remaining: 0


In [ ]:
# Install required packages
!pip install -r requirements.txt -q

## Step 2: Download and Compile Models

If you haven't compiled the models yet, run the compile script:

In [3]:
%%bash
cd /home/ubuntu/hf_pretrained_qwen_image_edit

echo "Downloading Qwen/Qwen-Image-Edit-2511..."
echo "Cache dir: /opt/dlami/nvme/qwen_image_edit_hf_cache_dir"
echo "This may take 10-20 minutes for ~58 GB."
echo ""

time python neuron_qwen_image_edit/cache_hf_model.py

echo ""
echo "=== Cache dir size ==="
du -sh /opt/dlami/nvme/qwen_image_edit_hf_cache_dir/
echo ""
echo "=== Disk space remaining ==="
df -h / /opt/dlami/nvme 2>/dev/null || df -h /

Cache dir: /opt/dlami/nvme/qwen_image_edit_hf_cache_dir
This may take 10-20 minutes for ~58 GB.



Loading pipeline components...: 100%|██████████| 6/6 [00:03<00:00,  1.67it/s]


Model downloaded successfully!



real	4m25.098s
user	1m32.799s
sys	2m25.925s



=== Cache dir size ===
54G	/opt/dlami/nvme/qwen_image_edit_hf_cache_dir/

=== Disk space remaining ===
Filesystem      Size  Used Avail Use% Mounted on
/dev/root       484G   99G  385G  21% /
/dev/root       484G   99G  385G  21% /


In [1]:
# Compile the models (this may take a while)
!sh compile.sh v3_tp16 1024 512 448 8 1024 3 1

compile.sh: 35: [[: not found
compile.sh: 35: v3_tp16: not found
compile.sh: 35: v3_tp16: not found
compile.sh: 35: v3_tp16: not found
compile.sh: 35: v3_tp16: not found
compile.sh: 35: v3_tp16: not found
compile.sh: 35: v3_tp16: not found
compile.sh: 35: v3_tp16: not found
compile.sh: 35: v3_tp16: not found
compile.sh: 41: [[: not found
compile.sh: 43: [[: not found
compile.sh: 45: [[: not found
Qwen-Image-Edit-2511 Compilation for Neuron
Transformer Version: all
Output Size: v3_tp16x1024
VAE Tile Size: 512x512 (fixed)
Vision Encoder Image Size: 512
TP Degree: 448
Max Sequence Length: 8
Patch Multiplier: 1024
Batch Size: 3

[Step 1/4] Downloading model...
Loading pipeline components...: 100%|█████████████| 6/6 [00:01<00:00,  5.22it/s]
Model downloaded successfully!
Model downloaded successfully!

[Step 2/4] Compiling VAE...
VAE tile size: 512x512 (tiled processing for larger images)
Using modified VAE with 'nearest' interpolation (Neuron doesn't support 'nearest-exact')
NKI Flash Atte

## Step 3: Load Pipeline and Compiled Models

In [2]:
import os
import torch
import torch_neuronx
import neuronx_distributed
from PIL import Image
from diffusers import QwenImageEditPlusPipeline

from neuron_qwen_image_edit.neuron_commons import (
    InferenceTransformerWrapper,
    SimpleWrapper,
)

/opt/aws_neuronx_venv_pytorch_2_9_nxd_inference/lib/python3.12/site-packages/neuronx_distributed/parallel_layers/layers.py:16: DeprecationWarning: torch_neuronx.nki_jit is deprecated, use nki.jit instead.
  from .mappings import (
/opt/aws_neuronx_venv_pytorch_2_9_nxd_inference/lib/python3.12/site-packages/neuronx_distributed/parallel_layers/layers.py:16: DeprecationWarning: torch_neuronx.nki_jit is deprecated, use nki.jit instead.
  from .mappings import (
/opt/aws_neuronx_venv_pytorch_2_9_nxd_inference/lib/python3.12/site-packages/neuronx_distributed/parallel_layers/layers.py:16: DeprecationWarning: torch_neuronx.nki_jit is deprecated, use nki.jit instead.
  from .mappings import (
/opt/aws_neuronx_venv_pytorch_2_9_nxd_inference/lib/python3.12/site-packages/neuronx_distributed/modules/moe/blockwise.py:74: DeprecationWarning: torch_neuronx.nki_jit is deprecated, use nki.jit instead.
  component, error = import_nki(config)
/opt/aws_neuronx_venv_pytorch_2_9_nxd_inference/lib/python3.12/

NKI Flash Attention kernel loaded successfully


In [4]:
# Configuration
CACHE_DIR = "/opt/dlami/nvme/qwen_image_edit_hf_cache_dir"
MODEL_ID = "Qwen/Qwen-Image-Edit-2511"
COMPILED_MODELS_DIR = "/opt/dlami/nvme/compiled_models"

# Load the pipeline
print("Loading pipeline...")
pipe = QwenImageEditPlusPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    local_files_only=True,
    cache_dir=CACHE_DIR
)
print("Pipeline loaded!")

Loading pipeline...


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

Pipeline loaded!


In [5]:
# Load compiled Neuron models
print("Loading compiled Neuron models...")

# Load compiled transformer
transformer_path = f"{COMPILED_MODELS_DIR}/transformer"
if os.path.exists(transformer_path):
    print(f"Loading compiled transformer...")
    transformer_wrapper = InferenceTransformerWrapper(pipe.transformer)
    transformer_wrapper.transformer = neuronx_distributed.trace.parallel_model_load(
        transformer_path
    )
    pipe.transformer = transformer_wrapper
    print("Transformer loaded!")

# Load compiled VAE decoder
decoder_path = f"{COMPILED_MODELS_DIR}/vae_decoder/model.pt"
if os.path.exists(decoder_path):
    print(f"Loading compiled VAE decoder...")
    vae_decoder_wrapper = SimpleWrapper(pipe.vae.decoder)
    vae_decoder_wrapper.model = torch_neuronx.DataParallel(
        torch.jit.load(decoder_path), [0, 1, 2, 3], False
    )
    pipe.vae.decoder = vae_decoder_wrapper
    print("VAE decoder loaded!")

# Load compiled VAE encoder (if exists)
encoder_path = f"{COMPILED_MODELS_DIR}/vae_encoder/model.pt"
if os.path.exists(encoder_path):
    print(f"Loading compiled VAE encoder...")
    vae_encoder_wrapper = SimpleWrapper(pipe.vae.encoder)
    vae_encoder_wrapper.model = torch_neuronx.DataParallel(
        torch.jit.load(encoder_path), [0, 1, 2, 3], False
    )
    pipe.vae.encoder = vae_encoder_wrapper
    print("VAE encoder loaded!")

print("All models loaded!")

Loading compiled Neuron models...
Loading compiled VAE decoder...


/opt/aws_neuronx_venv_pytorch_2_9_nxd_inference/lib/python3.12/site-packages/torch_neuronx/xla_impl/data_parallel.py:213: UserWarning: Found a weight separated model, setting dynamic_batching to False
  warnings.warn("Found a weight separated model, setting dynamic_batching to False")
Neuron: Dynamic batching is disabled for torch_neuronx.DataParallel
Neuron: Dynamic batching is disabled for torch_neuronx.DataParallel


VAE decoder loaded!
Loading compiled VAE encoder...
VAE encoder loaded!
All models loaded!


In [6]:
!sh ./test_tp16.sh

./test_tp16.sh: 1: source: not found
Neuron runtime configured: world_size=32, LNC=2
/opt/aws_neuronx_venv_pytorch_2_9_nxd_inference/lib/python3.12/site-packages/neuronx_distributed/parallel_layers/layers.py:16: DeprecationWarning: torch_neuronx.nki_jit is deprecated, use nki.jit instead.
  from .mappings import (
/opt/aws_neuronx_venv_pytorch_2_9_nxd_inference/lib/python3.12/site-packages/neuronx_distributed/parallel_layers/layers.py:16: DeprecationWarning: torch_neuronx.nki_jit is deprecated, use nki.jit instead.
  from .mappings import (
/opt/aws_neuronx_venv_pytorch_2_9_nxd_inference/lib/python3.12/site-packages/neuronx_distributed/parallel_layers/layers.py:16: DeprecationWarning: torch_neuronx.nki_jit is deprecated, use nki.jit instead.
  from .mappings import (
/opt/aws_neuronx_venv_pytorch_2_9_nxd_inference/lib/python3.12/site-packages/neuronx_distributed/modules/moe/blockwise.py:74: DeprecationWarning: torch_neuronx.nki_jit is deprecated, use nki.jit instead.
  component, error

## Step 4: Run Inference

### Example 1: Text-to-Image Generation

In [ ]:
# Text-to-image generation
prompt = "A beautiful sunset over mountains with a lake in the foreground, photorealistic"

inputs = {
    "prompt": prompt,
    "negative_prompt": " ",
    "num_inference_steps": 40,
    "guidance_scale": 1.0,
    "true_cfg_scale": 4.0,
    "generator": torch.manual_seed(42),
    "num_images_per_prompt": 1,
}

# Warmup run
print("Running warmup...")
with torch.inference_mode():
    _ = pipe(**inputs)
print("Warmup complete!")

# Actual inference
import time
print("Running inference...")
start_time = time.time()
with torch.inference_mode():
    output = pipe(**inputs)
inference_time = time.time() - start_time
print(f"Inference completed in {inference_time:.2f}s")

# Display result
output.images[0]

### Example 2: Image Editing (Single Image)

In [ ]:
# Load an input image for editing
# Replace with your own image path
# input_image = Image.open("your_image.png")

# For demonstration, create a simple test image
import numpy as np
test_image = Image.fromarray(np.random.randint(0, 255, (512, 512, 3), dtype=np.uint8))

prompt = "Transform this image into a watercolor painting style"

inputs = {
    "image": [test_image],
    "prompt": prompt,
    "negative_prompt": " ",
    "num_inference_steps": 40,
    "guidance_scale": 1.0,
    "true_cfg_scale": 4.0,
    "generator": torch.manual_seed(0),
    "num_images_per_prompt": 1,
}

print("Running image editing...")
with torch.inference_mode():
    output = pipe(**inputs)

# Display result
output.images[0]

### Example 3: Multi-Image Editing

In [ ]:
# Multi-image editing (combining 2 images)
# Replace with your own images
# image1 = Image.open("person.png")
# image2 = Image.open("background.png")

# For demonstration, create test images
image1 = Image.fromarray(np.random.randint(0, 255, (512, 512, 3), dtype=np.uint8))
image2 = Image.fromarray(np.random.randint(0, 255, (512, 512, 3), dtype=np.uint8))

prompt = "Combine these two images: place the subject from the first image into the scene from the second image"

inputs = {
    "image": [image1, image2],
    "prompt": prompt,
    "negative_prompt": " ",
    "num_inference_steps": 40,
    "guidance_scale": 1.0,
    "true_cfg_scale": 4.0,
    "generator": torch.manual_seed(0),
    "num_images_per_prompt": 1,
}

print("Running multi-image editing...")
with torch.inference_mode():
    output = pipe(**inputs)

# Display result
output.images[0]

## Step 5: Save Results

In [ ]:
# Save the generated image
output.images[0].save("generated_image.png")
print("Image saved to generated_image.png")

## Notes

- The text encoder (Qwen2.5-VL) runs on CPU due to its complexity as a multimodal model
- The transformer and VAE are compiled for Neuron and run on Trainium2
- First inference includes warmup time; subsequent inferences will be faster
- For best performance, use batch sizes that match the compiled configuration